<a href="https://colab.research.google.com/github/mayank261193/Rebounce-Applied-AI-and-Analytics-Codes/blob/main/MP9_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MP9 · Ask the Database — SQL on India's districts (Census 2011 + Aspirational Districts)
Four linked tables: `states`, `districts`, `household_amenities`, `aspirational_districts`. Each question has a text cell with the question number and a one-line answer, followed by one code cell with the query.

In [ ]:
import os, sqlite3, pandas as pd
from IPython.display import display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DB = "mp9_india_districts.db"
if not os.path.exists(DB):
    try:
        from google.colab import files
        files.upload()                      # pick mp9_india_districts.db
    except ImportError:
        DB = "/mnt/user-data/uploads/mp9_india_districts.db"
con = sqlite3.connect(DB)

def q(sql):
    return pd.read_sql(sql, con)

for t in ["states", "districts", "household_amenities", "aspirational_districts"]:
    print(t, q(f"SELECT COUNT(*) AS n FROM {t}").iloc[0, 0], "rows")

states 35 rows
districts 640 rows
household_amenities 640 rows
aspirational_districts 110 rows


### Q1. Show all columns of the states table.
**Answer:** The states table has 35 rows with state_id, state_name (Census 2011 names in capitals) and region.

In [ ]:
# Q1
display(q("""SELECT * FROM states;"""))

,state_id,state_name,region
0,1,JAMMU AND KASHMIR,North
1,2,HIMACHAL PRADESH,North
2,3,PUNJAB,North
3,4,CHANDIGARH,North
4,5,UTTARAKHAND,North
5,6,HARYANA,North
6,7,NCT OF DELHI,North
7,8,RAJASTHAN,North
8,9,UTTAR PRADESH,Central
9,10,BIHAR,East


### Q2. List the different regions, each once.
**Answer:** There are 6 regions: North, Central, East, North-East, West and South.

In [ ]:
# Q2
display(q("""SELECT DISTINCT region FROM states;"""))

,region
0,North
1,Central
2,East
3,North-East
4,West
5,South


### Q3. Show the 10 most populous districts, largest first.
**Answer:** Thane (about 110.6 lakh) is the most populous district, followed by North Twenty Four Parganas (100.1 lakh) and Bangalore (96.2 lakh).

In [ ]:
# Q3
display(q("""SELECT district_name, population
FROM districts
ORDER BY population DESC
LIMIT 10;"""))

,district_name,population
0,Thane,11060148
1,North Twenty Four Parganas,10009781
2,Bangalore,9621551
3,Pune,9429408
4,Mumbai Suburban,9356962
5,South Twenty Four Parganas,8161961
6,Barddhaman,7717563
7,Ahmadabad,7214225
8,Murshidabad,7103807
9,Jaipur,6626178


### Q4. How many districts have a population under 1 lakh (100,000)? List them.
**Answer:** 26 districts have a population under 1 lakh. Dibang Valley is the smallest, with 8,004 people. Most are in the North-East and in small island or Himalayan districts.

In [ ]:
# Q4
display(q("""SELECT COUNT(*) AS districts_under_1_lakh FROM districts WHERE population < 100000;"""))
display(q("""SELECT district_name, population
FROM districts
WHERE population < 100000
ORDER BY population;"""))

,districts_under_1_lakh
0,26


,district_name,population
0,Dibang Valley,8004
1,Anjaw,21167
2,Lahul AND Spiti,31564
3,Upper Siang,35320
4,Nicobars,36842
5,Mahe,41816
6,North District,43709
7,Tawang,49977
8,Longleng,50484
9,Diu,52074


### Q5. List the states in the North-East region.
**Answer:** The North-East region has 8 states: Sikkim, Arunachal Pradesh, Nagaland, Manipur, Mizoram, Tripura, Meghalaya and Assam.

In [ ]:
# Q5
display(q("""SELECT state_name FROM states WHERE region = 'North-East';"""))

,state_name
0,SIKKIM
1,ARUNACHAL PRADESH
2,NAGALAND
3,MANIPUR
4,MIZORAM
5,TRIPURA
6,MEGHALAYA
7,ASSAM


### Q6. List districts with a population between 20 lakh and 30 lakh.
**Answer:** 106 districts have a population between 20 and 30 lakh (Begusarai at 29.7 lakh is the largest in this band).

In [ ]:
# Q6
display(q("""SELECT district_name, population
FROM districts
WHERE population BETWEEN 2000000 AND 3000000
ORDER BY population DESC;"""))
display(q("""SELECT COUNT(*) AS n FROM districts WHERE population BETWEEN 2000000 AND 3000000;"""))

,district_name,population
0,Begusarai,2970541
1,Sri Potti Sriramulu Nellore,2963557
2,Rohtas,2959918
3,Puruliya,2930115
4,Ranchi,2914253
...,...,...
101,Banka,2034763
102,Gwalior,2032036
103,Pilibhit,2031007
104,Firozpur,2029074


,n
0,106


### Q7. List districts whose name ends in "pur". How many are there?
**Answer:** 64 districts have names ending in "pur" (for example Alirajpur, Bhagalpur, Udaipur). SQLite's LIKE ignores case, so 'Pur' and 'pur' both match.

In [ ]:
# Q7
display(q("""SELECT COUNT(*) AS districts_ending_pur FROM districts WHERE district_name LIKE '%pur';"""))
display(q("""SELECT district_name FROM districts WHERE district_name LIKE '%pur' ORDER BY district_name;"""))

,districts_ending_pur
0,64


,district_name
0,Alirajpur
1,Anantapur
2,Anuppur
3,Balrampur
4,Bhagalpur
...,...
59,Sultanpur
60,Tiruppur
61,Udaipur
62,Udhampur


### Q8. Each district's literacy percentage; the 10 lowest.
**Answer:** The lowest literacy is in Alirajpur (28.8%), then Bijapur (34.1%), Jhabua (34.3%), Dakshin Bastar Dantewada (35.8%) and Shrawasti (37.9%). All 10 are below 40%.

In [ ]:
# Q8
display(q("""SELECT district_name, population, literate,
       100.0 * literate / population AS literacy_pct
FROM districts
ORDER BY literacy_pct ASC
LIMIT 10;"""))

,district_name,population,literate,literacy_pct
0,Alirajpur,728999,209754,28.772879
1,Bijapur,255230,86919,34.055166
2,Jhabua,1025048,352081,34.347757
3,Dakshin Bastar Dantewada,533638,191237,35.836466
4,Shrawasti,1117361,423313,37.885070
5,Nabarangapur,1220946,470379,38.525782
6,Pakur,900422,352881,39.190624
7,Barwani,1385881,549926,39.680607
8,Kurung Kumey,92076,36606,39.756288
9,Malkangiri,613192,244706,39.906913


### Q9. Female literacy percentage; the 10 lowest. Are they the same districts as in Q8?
**Answer:** No, the bottom 10 are not identical: 7 of 10 districts appear in both lists. Mewat, Ramban and Balrampur are in the female bottom 10 but not in the overall bottom 10, and female literacy is lower everywhere (Alirajpur is 24.2%).

In [ ]:
# Q9
display(q("""SELECT district_name, female_population, female_literate,
       100.0 * female_literate / female_population AS female_literacy_pct
FROM districts
ORDER BY female_literacy_pct ASC
LIMIT 10;"""))
display(q("""-- Overlap check: districts that appear in both bottom-10 lists
SELECT district_name FROM (SELECT district_name FROM districts ORDER BY 100.0*literate/population ASC LIMIT 10)
INTERSECT
SELECT district_name FROM (SELECT district_name FROM districts ORDER BY 100.0*female_literate/female_population ASC LIMIT 10);"""))

,district_name,female_population,female_literate,female_literacy_pct
0,Alirajpur,366457,88849,24.245409
1,Bijapur,126567,32843,25.949102
2,Jhabua,510025,137499,26.959267
3,Dakshin Bastar Dantewada,269496,74708,27.721376
4,Shrawasti,523464,146613,28.008230
5,Mewat,518101,146462,28.269005
6,Nabarangapur,616134,183395,29.765441
7,Ramban,134581,41127,30.559291
8,Balrampur,1033944,323414,31.279644
9,Malkangiri,309568,97705,31.561725


,district_name
0,Alirajpur
1,Bijapur
2,Dakshin Bastar Dantewada
3,Jhabua
4,Malkangiri
5,Nabarangapur
6,Shrawasti


### Q10. Total population of all districts, and the average district population.
**Answer:** The total population is 1,210,854,977 (about 121 crore) across 640 districts, an average of about 18.9 lakh (1,891,961) per district.

In [ ]:
# Q10
display(q("""SELECT SUM(population) AS total_population,
       AVG(population) AS avg_district_population,
       COUNT(*)        AS n_districts
FROM districts;"""))

,total_population,avg_district_population,n_districts
0,1210854977,1.891961e+06,640


### Q11. How many states are in each region? Most to fewest.
**Answer:** North-East and North each have 8 states, South has 7, West 5, East 4 and Central 3.

In [ ]:
# Q11
display(q("""SELECT region, COUNT(*) AS n_states
FROM states
GROUP BY region
ORDER BY n_states DESC;"""))

,region,n_states
0,North-East,8
1,North,8
2,South,7
3,West,5
4,East,4
5,Central,3


### Q12. Which districts have more than 90% rural households? Count them.
**Answer:** 103 districts have more than 90% rural households. Lahul and Spiti, Kinnaur and Nicobars are 100% rural.

In [ ]:
# Q12
display(q("""SELECT COUNT(*) AS districts_over_90pct_rural
FROM districts
WHERE rural_households * 100.0 / households > 90;"""))
display(q("""SELECT district_name, rural_households * 100.0 / households AS rural_pct
FROM districts
WHERE rural_households * 100.0 / households > 90
ORDER BY rural_pct DESC
LIMIT 15;"""))

,districts_over_90pct_rural
0,103


,district_name,rural_pct
0,Lahul AND Spiti,100.000000
1,Kinnaur,100.000000
2,Nicobars,100.000000
3,Baksa,98.787349
4,North AND Middle Andaman,97.277591
5,Shrawasti,96.694575
6,Madhubani,96.212846
7,Rudraprayag,96.198329
8,Bageshwar,96.116544
9,Samastipur,95.883820


### Q13. The 10 districts with the lowest literacy, with state name.
**Answer:** The lowest-literacy districts are in Madhya Pradesh (Alirajpur, Jhabua, Barwani), Chhattisgarh (Bijapur, Dakshin Bastar Dantewada), Uttar Pradesh (Shrawasti), Orissa (Nabarangapur, Malkangiri), Jharkhand (Pakur) and Arunachal Pradesh (Kurung Kumey).

In [ ]:
# Q13
display(q("""SELECT d.district_name, s.state_name,
       100.0 * d.literate / d.population AS literacy_pct
FROM districts d
INNER JOIN states s ON d.state_id = s.state_id
ORDER BY literacy_pct ASC
LIMIT 10;"""))

,district_name,state_name,literacy_pct
0,Alirajpur,MADHYA PRADESH,28.772879
1,Bijapur,CHHATTISGARH,34.055166
2,Jhabua,MADHYA PRADESH,34.347757
3,Dakshin Bastar Dantewada,CHHATTISGARH,35.836466
4,Shrawasti,UTTAR PRADESH,37.885070
5,Nabarangapur,ORISSA,38.525782
6,Pakur,JHARKHAND,39.190624
7,Barwani,MADHYA PRADESH,39.680607
8,Kurung Kumey,ARUNACHAL PRADESH,39.756288
9,Malkangiri,ORISSA,39.906913


### Q14. For each state: number of districts and total population. Top 5 by population.
**Answer:** The top 5 states by population are Uttar Pradesh (71 districts, 199.8 million), Maharashtra (35, 112.4 million), Bihar (38, 104.1 million), West Bengal (19, 91.3 million) and Andhra Pradesh (23, 84.6 million; includes Telangana in Census 2011).

In [ ]:
# Q14
display(q("""SELECT s.state_name, COUNT(*) AS n_districts, SUM(d.population) AS total_population
FROM districts d
INNER JOIN states s ON d.state_id = s.state_id
GROUP BY s.state_name
ORDER BY total_population DESC
LIMIT 5;"""))

,state_name,n_districts,total_population
0,UTTAR PRADESH,71,199812341
1,MAHARASHTRA,35,112374333
2,BIHAR,38,104099452
3,WEST BENGAL,19,91276115
4,ANDHRA PRADESH,23,84580777


### Q15. For each region: share of households with tap water (three tables).
**Answer:** Tap-water access is highest in the South (51.9% of households) and West (48.1%), and lowest in the East (11.6%) and North-East (15.2%). Shares are computed as SUM(tap water households) / SUM(households).

In [ ]:
# Q15
display(q("""SELECT s.region,
       100.0 * SUM(a.hh_tap_water) / SUM(d.households) AS tap_water_pct
FROM districts d
INNER JOIN states s ON d.state_id = s.state_id
INNER JOIN household_amenities a ON d.district_code = a.district_code
GROUP BY s.region
ORDER BY tap_water_pct DESC;"""))

,region,tap_water_pct
0,South,51.933853
1,West,48.088773
2,North,37.686835
3,Central,19.347565
4,North-East,15.224448
5,East,11.640025


### Q16. The 10 districts with the lowest share of households with internet, with state name.
**Answer:** The 10 districts with the lowest internet access all have under 0.25% of households online, from Pakur (0.15%) and Sheopur (0.15%) to Sahibganj (0.23%). Jharkhand has four of them.

In [ ]:
# Q16
display(q("""SELECT d.district_name, s.state_name,
       100.0 * a.hh_internet / d.households AS internet_pct
FROM districts d
INNER JOIN states s ON d.state_id = s.state_id
INNER JOIN household_amenities a ON d.district_code = a.district_code
ORDER BY internet_pct ASC
LIMIT 10;"""))

,district_name,state_name,internet_pct
0,Pakur,JHARKHAND,0.151030
1,Sheopur,MADHYA PRADESH,0.154655
2,West Khasi Hills,MEGHALAYA,0.171007
3,Baudh,ORISSA,0.177595
4,Garhwa,JHARKHAND,0.193459
5,Alirajpur,MADHYA PRADESH,0.197305
6,Bijapur,CHHATTISGARH,0.200373
7,Latehar,JHARKHAND,0.211820
8,Panna,MADHYA PRADESH,0.226012
9,Sahibganj,JHARKHAND,0.226443


### Q17. Every Aspirational District with its state and literacy %, lowest first.
**Answer:** There are 110 Aspirational Districts. Literacy ranges from 34.1% (Bijapur, Chhattisgarh) to 79.0% (Wayanad, Kerala). Most of the lowest are in Chhattisgarh, Jharkhand, Orissa and Uttar Pradesh, but a few have high literacy, such as Wayanad and Dhalai.

In [ ]:
# Q17
display(q("""SELECT ad.district_name, s.state_name,
       100.0 * d.literate / d.population AS literacy_pct
FROM aspirational_districts ad
INNER JOIN districts d ON ad.district_code = d.district_code
INNER JOIN states s ON d.state_id = s.state_id
ORDER BY literacy_pct ASC;"""))

,district_name,state_name,literacy_pct
0,Bijapur,CHHATTISGARH,34.055166
1,Dakshin Bastar Dantewada,CHHATTISGARH,35.836466
2,Shrawasti,UTTAR PRADESH,37.885070
3,Nabarangapur,ORISSA,38.525782
4,Pakur,JHARKHAND,39.190624
...,...,...,...
105,Virudhunagar,TAMIL NADU,72.017538
106,Ramanathapuram,TAMIL NADU,72.329943
107,Washim,MAHARASHTRA,72.665057
108,Dhalai,TRIPURA,73.028845


### Q18. Count Aspirational Districts per state; only states with 5 or more.
**Answer:** 9 states have 5 or more Aspirational Districts: Jharkhand 19, Bihar 13, Orissa 10, Uttar Pradesh 8, Madhya Pradesh 8, Chhattisgarh 8, Assam 7, Andhra Pradesh 6 and Rajasthan 5. Together these account for 84 of the 110 districts.

In [ ]:
# Q18
display(q("""SELECT s.state_name, COUNT(*) AS n_aspirational
FROM aspirational_districts ad
INNER JOIN districts d ON ad.district_code = d.district_code
INNER JOIN states s ON d.state_id = s.state_id
GROUP BY s.state_name
HAVING COUNT(*) >= 5
ORDER BY n_aspirational DESC;"""))

,state_name,n_aspirational
0,JHARKHAND,19
1,BIHAR,13
2,ORISSA,10
3,UTTAR PRADESH,8
4,MADHYA PRADESH,8
5,CHHATTISGARH,8
6,ASSAM,7
7,ANDHRA PRADESH,6
8,RAJASTHAN,5


### Q19. Using a LEFT JOIN, count the districts that are not Aspirational.
**Answer:** 530 districts are not Aspirational (640 total minus 110 listed).

In [ ]:
# Q19
display(q("""SELECT COUNT(*) AS non_aspirational_districts
FROM districts d
LEFT JOIN aspirational_districts ad ON d.district_code = ad.district_code
WHERE ad.asp_id IS NULL;"""))

,non_aspirational_districts
0,530


### Q20. Overall literacy % of Aspirational Districts (query 1) and of all other districts (query 2).
**Answer:** Aspirational Districts have an overall literacy of 53.4% against 64.6% for all other districts, a gap of about 11.2 percentage points. Both figures are computed as total literate divided by total population.

In [ ]:
# Q20
display(q("""-- 20a: Aspirational Districts
SELECT 100.0 * SUM(d.literate) / SUM(d.population) AS aspirational_literacy_pct,
       COUNT(*) AS n_districts
FROM districts d
INNER JOIN aspirational_districts ad ON d.district_code = ad.district_code;"""))
display(q("""-- 20b: all other districts
SELECT 100.0 * SUM(d.literate) / SUM(d.population) AS other_literacy_pct,
       COUNT(*) AS n_districts
FROM districts d
LEFT JOIN aspirational_districts ad ON d.district_code = ad.district_code
WHERE ad.asp_id IS NULL;"""))

,aspirational_literacy_pct,n_districts
0,53.375133,110


,other_literacy_pct,n_districts
0,64.606655,530


### Q21. Districts that are not Aspirational but have literacy below 50%. Could any have been missed?
**Answer:** 40 districts below 50% literacy are not Aspirational, including Alirajpur (28.8%, the lowest in the dataset) and Jhabua (34.3%). Bihar has 9 of them, Jammu and Kashmir 7, Uttar Pradesh 6, Rajasthan 5 and Arunachal Pradesh 5. Literacy alone would put several of these within the range of listed districts (37 listed districts are also below 50%), so some may have been missed, but the NITI list uses a broader set of indicators (health, education, agriculture, finance, infrastructure), so literacy cannot show who was wrongly left out.

In [ ]:
# Q21
display(q("""SELECT d.district_name, s.state_name,
       100.0 * d.literate / d.population AS literacy_pct,
       100.0 * a.hh_internet / d.households AS internet_pct
FROM districts d
LEFT JOIN aspirational_districts ad ON d.district_code = ad.district_code
INNER JOIN states s ON d.state_id = s.state_id
INNER JOIN household_amenities a ON d.district_code = a.district_code
WHERE ad.asp_id IS NULL
  AND 100.0 * d.literate / d.population < 50
ORDER BY literacy_pct ASC;"""))
display(q("""-- context: highest literacy among Aspirational Districts (to judge whether the non-listed ones look comparable)
SELECT MAX(100.0 * d.literate / d.population) AS max_aspirational_literacy,
       MIN(100.0 * d.literate / d.population) AS min_aspirational_literacy
FROM districts d INNER JOIN aspirational_districts ad ON d.district_code = ad.district_code;"""))

,district_name,state_name,literacy_pct,internet_pct
0,Alirajpur,MADHYA PRADESH,28.772879,0.197305
1,Jhabua,MADHYA PRADESH,34.347757,0.315531
2,Kurung Kumey,ARUNACHAL PRADESH,39.756288,0.482356
3,Madhepura,BIHAR,41.692119,0.273772
4,Budaun,UTTAR PRADESH,42.029351,0.418845
5,Saharsa,BIHAR,42.353476,0.354368
6,Tirap,ARUNACHAL PRADESH,42.843492,0.393258
7,Sheohar,BIHAR,43.230435,0.287952
8,Ramban,JAMMU AND KASHMIR,43.729050,0.465288
9,Kishanganj,BIHAR,44.075722,0.245535


,max_aspirational_literacy,min_aspirational_literacy
0,78.978371,34.055166


## Policy note
Aspirational Districts are clearly behind the rest of the country: their overall literacy is 53.4% against 64.6% in the other 530 districts, and in the same 2011 data only 0.9% of their households have internet compared with 2.5% elsewhere. The problem is concentrated: Jharkhand (19 districts) and Bihar (13) alone hold 32 of the 110 listed districts, and the lowest literacy is in a few tribal and remote districts such as Alirajpur (28.8%), Bijapur (34.1%) and Jhabua (34.3%). Region matters too: only 11.6% of households in the East and 15.2% in the North-East have tap water, compared with 51.9% in the South. The main limitation is age: these are Census 2011 figures on 2011 boundaries, and the Aspirational list was drawn up in 2018 using wider indicators, so literacy and amenities have changed since then. 40 districts that are not on the list still have literacy below 50% (including Alirajpur and Jhabua in Madhya Pradesh, and nine in Bihar), so I suggest the state review those districts against current data and consider adding them to a monitoring list.